# 01 · Ingesta y verificación de datos

**Fase 1 del E3.** Convertimos los 9 CSV del SCANIA Component X Dataset a Parquet, cargamos cada conjunto (lecturas + especificaciones + etiquetas) y comprobamos que los datos son coherentes antes de limpiarlos y explorarlos.

Comprobaciones:
1. Los 9 ficheros están completos (filas esperadas).
2. Cada `vehicle_id` aparece en los tres ficheros de su conjunto y no hay camiones compartidos entre train, validación y test.
3. No hay lecturas duplicadas y `time_step` crece dentro de cada camión.
4. Porcentaje de nulos por columna.
5. Contadores que decrecen (posibles reinicios de la ECU).
6. Distribución de etiquetas y coste de referencia "predecir siempre 0".

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import data

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Ficheros originales

In [2]:
raw_status = data.check_raw(count_rows=True)
raw_status

,fichero,existe,MB,filas,esperadas,ok
0,train_operational_readouts.csv,True,"1,219.20",1122452,1122452,True
1,train_tte.csv,True,0.30,23550,23550,True
2,train_specifications.csv,True,1.10,23550,23550,True
3,validation_operational_readouts.csv,True,215.60,196227,196227,True
4,validation_labels.csv,True,0.00,5046,5046,True
5,validation_specifications.csv,True,0.20,5046,5046,True
6,test_operational_readouts.csv,True,214.90,198140,198140,True
7,test_labels.csv,True,0.00,5045,5045,True
8,test_specifications.csv,True,0.20,5045,5045,True


In [3]:
assert raw_status["ok"].all(), "Falta algún CSV o no tiene las filas esperadas" 

## 2. Conversión a Parquet

Se hace por bloques para no cargar el CSV de entrenamiento (1,14 GB) entero en memoria. Las lecturas se guardan en `float64` porque algunos contadores superan 2²⁴ y en `float32` perderían precisión. Si los Parquet ya existen, no se regeneran.

In [4]:
data.build_parquet()

pd.DataFrame([
    {"fichero": p.name, "MB": round(p.stat().st_size / 1e6, 1)}
    for p in sorted(data.PROCESSED.glob("*.parquet"))
])

  ya existe  train_operational_readouts.parquet
  ya existe  train_specifications.parquet
  ya existe  train_tte.parquet
  ya existe  validation_operational_readouts.parquet
  ya existe  validation_specifications.parquet
  ya existe  validation_labels.parquet
  ya existe  test_operational_readouts.parquet
  ya existe  test_specifications.parquet
  ya existe  test_labels.parquet


,fichero,MB
0,test_labels.parquet,0.00
1,test_operational_readouts.parquet,106.60
2,test_specifications.parquet,0.00
3,train_operational_readouts.parquet,600.20
4,train_specifications.parquet,0.10
5,train_tte.parquet,0.10
6,validation_labels.parquet,0.00
7,validation_operational_readouts.parquet,108.50
8,validation_specifications.parquet,0.00


In [5]:
csv_mb = raw_status["MB"].sum()
pq_mb = sum(p.stat().st_size for p in data.PROCESSED.glob("*.parquet")) / 1e6
print(f"CSV: {csv_mb:,.0f} MB  ->  Parquet: {pq_mb:,.0f} MB  ({pq_mb / csv_mb:.0%})")

CSV: 1,652 MB  ->  Parquet: 816 MB  (49%)


## 3. Carga de los tres conjuntos

Para las comprobaciones cargamos las lecturas en `float32` (la mitad de memoria); el análisis posterior puede usar `float64`.

In [6]:
splits = {s: data.load_split(s, float32=True) for s in data.SPLITS}

pd.DataFrame([
    {
        "conjunto": s,
        "camiones": r["vehicle_id"].nunique(),
        "lecturas": len(r),
        "columnas lecturas": r.shape[1],
        "lecturas/camión (mediana)": r.groupby("vehicle_id").size().median(),
        "memoria lecturas (MB)": round(r.memory_usage(deep=True).sum() / 1e6),
    }
    for s, (r, sp, lb) in splits.items()
])

,conjunto,camiones,lecturas,columnas lecturas,lecturas/camión (mediana),memoria lecturas (MB)
0,train,23550,1122452,107,43.00,485
1,validation,5046,196227,107,34.00,85
2,test,5045,198140,107,34.00,86


## 4. Coherencia de identificadores

In [7]:
rows = []
ids = {}
for s, (r, sp, lb) in splits.items():
    a, b, c = set(r["vehicle_id"]), set(sp["vehicle_id"]), set(lb["vehicle_id"])
    ids[s] = a
    rows.append({
        "conjunto": s,
        "ids lecturas": len(a), "ids specs": len(b), "ids etiquetas": len(c),
        "mismos ids en los 3": a == b == c,
        "ids duplicados en specs": sp["vehicle_id"].duplicated().sum(),
        "ids duplicados en etiquetas": lb["vehicle_id"].duplicated().sum(),
    })
id_check = pd.DataFrame(rows)
id_check

,conjunto,ids lecturas,ids specs,ids etiquetas,mismos ids en los 3,ids duplicados en specs,ids duplicados en etiquetas
0,train,23550,23550,23550,True,0,0
1,validation,5046,5046,5046,True,0,0
2,test,5045,5045,5045,True,0,0


In [8]:
print("train ∩ validation:", len(ids["train"] & ids["validation"]))
print("train ∩ test:      ", len(ids["train"] & ids["test"]))
print("validation ∩ test: ", len(ids["validation"] & ids["test"]))

train ∩ validation: 0
train ∩ test:       0
validation ∩ test:  0


## 5. Duplicados y orden temporal

In [9]:
rows = []
for s, (r, _, _) in splits.items():
    dup_rows = r.duplicated().sum()
    dup_key = r.duplicated(["vehicle_id", "time_step"]).sum()
    # tras ordenar, time_step debe ser estrictamente creciente dentro de cada camión
    step = r.groupby("vehicle_id")["time_step"].diff()
    rows.append({
        "conjunto": s,
        "filas duplicadas": dup_rows,
        "(vehicle_id, time_step) repetidos": dup_key,
        "saltos time_step <= 0": int((step <= 0).sum()),
        "time_step mín": r["time_step"].min(),
        "time_step máx": r["time_step"].max(),
    })
order_check = pd.DataFrame(rows)
order_check

,conjunto,filas duplicadas,"(vehicle_id, time_step) repetidos",saltos time_step <= 0,time_step mín,time_step máx
0,train,0,0,0,0.00,507.40
1,validation,0,0,0,0.00,456.60
2,test,0,0,0,0.00,447.60


## 6. Valores nulos

In [10]:
null_pct = pd.DataFrame({s: r[data.SENSOR_COLUMNS].isna().mean() * 100 for s, (r, _, _) in splits.items()})
print("Columnas con algún nulo:", (null_pct.max(axis=1) > 0).sum(), "de", len(null_pct))
print("Máximo % de nulos en una columna:", f"{null_pct.values.max():.2f} %")
null_pct.sort_values("train", ascending=False).head(10)

Columnas con algún nulo: 104 de 105
Máximo % de nulos en una columna: 0.90 %


,train,validation,test
291_3,0.86,0.81,0.90
291_9,0.86,0.81,0.90
291_10,0.86,0.81,0.90
291_8,0.86,0.81,0.90
291_1,0.86,0.81,0.90
291_4,0.86,0.81,0.90
291_7,0.86,0.81,0.90
291_6,0.86,0.81,0.90
291_5,0.86,0.81,0.90
291_0,0.86,0.81,0.90


In [11]:
# ¿Los nulos van por lectura completa (falta todo un sensor) o sueltos?
r = splits["train"][0]
null_by_var = pd.DataFrame({
    var: r[[c for c in data.SENSOR_COLUMNS if c.split("_")[0] == var]].isna().any(axis=1)
    for var in sorted({c.split("_")[0] for c in data.SENSOR_COLUMNS})
})
print(f"Lecturas de train con algún nulo: {null_by_var.any(axis=1).mean():.2%}")
null_by_var.mean().sort_values(ascending=False).mul(100).round(3).rename("% lecturas con nulos").to_frame()

Lecturas de train con algún nulo: 2.00%


,% lecturas con nulos
291,0.86
459,0.74
427,0.57
100,0.57
370,0.57
167,0.26
158,0.06
397,0.05
272,0.05
666,0.00


## 7. Contadores que decrecen

Los contadores son acumulados y deberían crecer siempre. Un descenso entre dos lecturas consecutivas del mismo camión apunta a un reinicio de la ECU o a un error de registro.

In [12]:
rows = []
for s, (r, _, _) in splits.items():
    diffs = r.groupby("vehicle_id")[data.COUNTERS].diff()
    dec = diffs < 0
    for c in data.COUNTERS:
        rows.append({
            "conjunto": s, "contador": c,
            "descensos": int(dec[c].sum()),
            "% lecturas": dec[c].mean() * 100,
            "camiones afectados": r.loc[dec[c], "vehicle_id"].nunique(),
        })
decreases = pd.DataFrame(rows).pivot(index="contador", columns="conjunto", values=["descensos", "camiones afectados"])
decreases

descensos                  camiones afectados                 
conjunto      test train validation               test train validation
contador                                                               
100_0            0     0          0                  0     0          0
171_0            0     7          0                  0     7          0
309_0            0     0          0                  0     0          0
370_0            0     0          0                  0     0          0
427_0            0     0          0                  0     0          0
666_0            0     0          0                  0     0          0
835_0            0     0          0                  0     0          0
837_0            0     0          0                  0     0          0

## 8. Etiquetas

### Entrenamiento: tiempo hasta el evento

In [13]:
tte = splits["train"][2]
print(tte["in_study_repair"].value_counts().rename({0: "sin reparación (censurado)", 1: "con reparación"}).to_frame("camiones")
      .assign(pct=lambda d: (d["camiones"] / d["camiones"].sum() * 100).round(2)))
tte.groupby("in_study_repair")["length_of_study_time_step"].describe()

                            camiones   pct
in_study_repair                           
sin reparación (censurado)     21278 90.35
con reparación                  2272  9.65


,count,mean,std,min,25%,50%,75%,max
in_study_repair,,,,,,,,
0,"21,278.00",243.16,89.18,121.00,165.20,219.00,316.40,510.00
1,"2,272.00",214.01,80.29,73.40,148.55,208.40,276.25,482.00


### Validación y test: clase de la última lectura

In [14]:
classes = pd.DataFrame({s: splits[s][2]["class_label"].value_counts().sort_index() for s in ["validation", "test"]})
classes.index.name = "class_label"
classes["% validación"] = (classes["validation"] / classes["validation"].sum() * 100).round(2)
classes["% test"] = (classes["test"] / classes["test"].sum() * 100).round(2)
classes

,validation,test,% validación,% test
class_label,,,,
0,4910,4903,97.30,97.19
1,16,26,0.32,0.52
2,14,15,0.28,0.30
3,30,41,0.59,0.81
4,76,60,1.51,1.19


### Coste de referencia: predecir siempre la clase 0

Es el coste de no avisar nunca (hipótesis H5 del E3). Cualquier modelo útil tiene que quedar por debajo.

In [15]:
def total_cost(y_true, y_pred, matrix=data.COST_MATRIX):
    m = np.asarray(matrix)
    return int(m[np.asarray(y_true), np.asarray(y_pred)].sum())

for s in ["validation", "test"]:
    y = splits[s][2]["class_label"]
    print(f"{s:<10} coste Scania: {total_cost(y, np.zeros_like(y)):>7,}   errores (matriz de 1s): {(y != 0).sum()}")

validation coste Scania:  57,400   errores (matriz de 1s): 136
test       coste Scania:  56,100   errores (matriz de 1s): 142


## 9. Lecturas por camión

In [16]:
per_vehicle = pd.DataFrame({s: r.groupby("vehicle_id").size().describe() for s, (r, _, _) in splits.items()}).T
per_vehicle

,count,mean,std,min,25%,50%,75%,max
train,"23,550.00",47.66,27.41,5.00,28.00,43.00,64.00,303.00
validation,"5,046.00",38.89,25.05,1.00,20.00,34.00,54.00,209.00
test,"5,045.00",39.27,25.61,1.00,20.00,34.00,55.00,262.00


In [17]:
r = splits["train"][0]
n_read = r.groupby("vehicle_id").size()
for k in [3, 5, 10, 20]:
    print(f"Camiones de train con menos de {k:>2} lecturas: {(n_read < k).sum():>5} ({(n_read < k).mean():.1%})")

Camiones de train con menos de  3 lecturas:     0 (0.0%)
Camiones de train con menos de  5 lecturas:     0 (0.0%)
Camiones de train con menos de 10 lecturas:   997 (4.2%)
Camiones de train con menos de 20 lecturas:  3341 (14.2%)


## 10. Resumen

Resultados de las comprobaciones (se rellenan al ejecutar):

In [18]:
r_tr = splits["train"][0]
summary = {
    "CSV completos": bool(raw_status["ok"].all()),
    "ids coherentes en los 3 ficheros": bool(id_check["mismos ids en los 3"].all()),
    "sin camiones compartidos entre conjuntos": not (ids["train"] & ids["validation"] or ids["train"] & ids["test"] or ids["validation"] & ids["test"]),
    "filas o (vehicle_id, time_step) duplicados": int(order_check[["filas duplicadas", "(vehicle_id, time_step) repetidos"]].values.sum()),
    "descensos de contadores en train": int(decreases["descensos"]["train"].sum()),
    "máx % nulos en una columna": round(float(null_pct.values.max()), 2),
    "camiones de train con reparación (%)": round(tte["in_study_repair"].mean() * 100, 2),
}
pd.Series(summary, name="valor").to_frame()

,valor
CSV completos,True
ids coherentes en los 3 ficheros,True
sin camiones compartidos entre conjuntos,True
"filas o (vehicle_id, time_step) duplicados",0
descensos de contadores en train,7
máx % nulos en una columna,0.90
camiones de train con reparación (%),9.65
